# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mashfiqmahi/assignment_FLyRank-AI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Building a richer feature vector than the 5-feature version: adds categorical handling
(content_type) and explicit missingness flags instead of silently filling with 0,
since missingness here follows content_type, not randomness.

In [1]:
%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, duckdb, pandas as pd, numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

raw = con.sql("""
WITH monthly AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE) AS impressions,
        SUM(gsc_clicks)      FILTER (WHERE gsc_data_available IS TRUE) AS clicks,
        SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE) AS sum_position,
        SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS engaged_sessions,
        SUM(ga4_sessions)         FILTER (WHERE ga4_data_available IS TRUE) AS sessions,
        BOOL_OR(ga4_data_available) AS ga4_available_flag
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
)
SELECT
    m.content_hash_id,
    m.impressions,
    m.ga4_available_flag,
    CASE WHEN m.impressions > 0 THEN m.clicks * 100.0 / m.impressions ELSE NULL END AS ctr_pct,
    CASE WHEN m.impressions > 0 THEN m.sum_position * 1.0 / m.impressions ELSE NULL END AS avg_position,
    CASE WHEN m.sessions > 0 THEN m.engaged_sessions * 100.0 / m.sessions ELSE NULL END AS engagement_rate_pct,
    DATE_DIFF('day', d.content_created_date, DATE '2026-03-31') AS days_since_created,
    d.content_type,
    d.word_count,
    d.backlinks
FROM monthly m
JOIN read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet') d
    ON m.content_hash_id = d.content_hash_id
""").df()

# Clip the small (~0.6%) future-dated anomaly instead of dropping those rows
raw["days_since_created"] = raw["days_since_created"].clip(lower=0)

# Flags instead of blind fillna — missingness is informative, not noise
raw["has_word_count"] = raw["word_count"].notna().astype(int)
raw["has_engagement_data"] = raw["ga4_available_flag"].fillna(False).astype(int)

content_type_dummies = pd.get_dummies(raw["content_type"], prefix="type")
X = pd.concat([raw, content_type_dummies], axis=1)

print(X.shape)
X.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 15)


,content_hash_id,impressions,ga4_available_flag,ctr_pct,avg_position,engagement_rate_pct,days_since_created,content_type,word_count,backlinks,has_word_count,has_engagement_data,type_comparison article,type_feedly article,type_keyword article
0,content_39d7361b4945d504,77.0,<NA>,0.000000,4.311688,NaN,47,keyword article,3579,0,1,0,False,False,True
1,content_cec711b02f3bbde6,602.0,<NA>,0.664452,4.365449,NaN,47,keyword article,2455,0,1,0,False,False,True
2,content_275b6f7f733016d4,810.0,<NA>,0.123457,4.624691,NaN,47,keyword article,3653,0,1,0,False,False,True
3,content_ceaec531566ffcfc,82.0,<NA>,0.000000,8.097561,NaN,47,keyword article,3096,0,1,0,False,False,True
4,content_755d951187fcd70a,1858.0,<NA>,0.322928,1.920344,NaN,47,keyword article,3305,27,1,0,False,False,True


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
missingness = X[["impressions","ctr_pct","avg_position","engagement_rate_pct",
                  "days_since_created","word_count","backlinks"]].isna().mean().round(3)
print("Overall missing rate per column:")
print(missingness)

Overall missing rate per column:
impressions            0.467
ctr_pct                0.467
avg_position           0.467
engagement_rate_pct    0.728
days_since_created     0.000
word_count             0.324
backlinks              0.511
dtype: float64


In [3]:
missing_by_type = X.groupby("content_type")[["word_count","engagement_rate_pct","backlinks"]].apply(
    lambda g: g.isna().mean()
).round(3)
print("\nMissing rate by content_type:")
print(missing_by_type)


Missing rate by content_type:
                    word_count  engagement_rate_pct  backlinks
content_type                                                  
comparison article       0.001                0.382      0.001
feedly article           0.057                0.836      1.000
keyword article          0.377                0.712      0.427


In [4]:
X["has_backlinks"] = X["backlinks"].notna().astype(int)

In [5]:
X[["has_word_count","has_backlinks","has_engagement_data"]].mean()

,0
has_word_count,0.675869
has_backlinks,0.488956
has_engagement_data,0.273020


Missingness overall (content-item level, aggregated over March):
- impressions/ctr_pct/avg_position: 46.7% missing — pages with zero GSC-available days this month.
- engagement_rate_pct: 72.8% missing — pages with zero GA4-available days this month.
  (Note: this is a page-level figure; the data contract notebook's 4.2% GA4 figure was
  row-level, i.e. per page-day. Same underlying sparsity, different denominator.)
- word_count: 32.4% missing overall, but NOT random — 37.7% missing for keyword articles
  vs 0.1% for comparison articles. Missingness follows content_type.
- backlinks: 51.1% missing overall, and 100% missing for feedly articles specifically —
  a structural gap (feedly-sourced content is never backlink-tracked), not noise.

Because of this pattern, word_count, backlinks, and engagement_rate_pct are NOT filled
with fillna(0) — that would fabricate a false "zero" signal tied to content_type. Instead,
each has a has_X flag (has_word_count, has_backlinks, has_engagement_data) so the model
can see "not measured" as distinct from "measured as zero."

Per-feature summary:
- impressions, ctr_pct, avg_position: numeric, feature, available at decision time
  (observed search performance this month).
- engagement_rate_pct: numeric, feature, available at decision time; sparse (72.8%
  missing at page level) — paired with has_engagement_data.
- days_since_created: numeric, feature, always available, 0% missing.
- word_count, backlinks: numeric, feature, available at decision time, but missingness
  follows content_type — paired with has_word_count / has_backlinks flags.
- content_type: categorical, feature, one-hot encoded (type_comparison article,
  type_feedly article, type_keyword article), always available.
- content_hash_id: context only, never a feature.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
excluded_columns = ["trend_direction", "trend_pct", "is_declining_label",  # label-derived (type 1)
                     "is_published", "is_deleted",                        # product/decision flags (type 3)
                     "keyword_hash_id", "url_hash_id"]                    # identifiers, not features

for col in excluded_columns:
    present = col in X.columns
    print(f"{col}: present in feature set? {present}")

trend_direction: present in feature set? False
trend_pct: present in feature set? False
is_declining_label: present in feature set? False
is_published: present in feature set? False
is_deleted: present in feature set? False
keyword_hash_id: present in feature set? False
url_hash_id: present in feature set? False


#### Click leaks

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

feature_cols = ["impressions","ctr_pct","avg_position","engagement_rate_pct",
                 "days_since_created","word_count","backlinks"] + list(content_type_dummies.columns)

clean = X.dropna(subset=["impressions","ctr_pct","avg_position","days_since_created"]).copy()
clean = clean[clean["impressions"] > 0]
clean["engagement_rate_pct"] = clean["engagement_rate_pct"].fillna(-1)  # sentinel: "not measured"
clean["word_count"] = clean["word_count"].fillna(-1)
clean["backlinks"] = clean["backlinks"].fillna(-1)

sample = clean.sample(n=min(20000, len(clean)), random_state=42)

# --- HONEST run ---
X_honest = sample[feature_cols].copy()
X_honest["impressions"] = np.log1p(X_honest["impressions"])
X_honest_scaled = StandardScaler().fit_transform(X_honest)
km_honest = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_honest_scaled)
sil_honest = silhouette_score(X_honest_scaled, km_honest.labels_)
print("Honest silhouette score:", sil_honest)

# --- LEAKY run: add a column that's just impressions*ctr restated ---
sample["clicks_leak"] = np.log1p(sample["impressions"] * sample["ctr_pct"] / 100)
X_leaky = pd.concat([X_honest, sample["clicks_leak"]], axis=1)
X_leaky_scaled = StandardScaler().fit_transform(X_leaky)
km_leaky = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_leaky_scaled)
sil_leaky = silhouette_score(X_leaky_scaled, km_leaky.labels_)
print("Leaky silhouette score:", sil_leaky)

print(f"\nGap: {sil_leaky - sil_honest:.3f}  <- this gap IS the leakage, made visible")

Honest silhouette score: 0.31166082275382195
Leaky silhouette score: 0.28499020388102747

Gap: -0.027  <- this gap IS the leakage, made visible


In [9]:
del sample["clicks_leak"]
print(f"Final, honest silhouette score kept for this feature set: {sil_honest:.3f}")

Final, honest silhouette score kept for this feature set: 0.312


In [12]:
corr = sample[["impressions", "ctr_pct", "clicks_leak"]].corr()
print(corr)

             impressions   ctr_pct  clicks_leak
impressions     1.000000 -0.014490     0.618533
ctr_pct        -0.014490  1.000000     0.057827
clicks_leak     0.618533  0.057827     1.000000


Correlation check: clicks_leak correlates strongly with impressions (r=0.62) but
weakly with ctr_pct (r=0.058) — ctr_pct's low variance (most pages cluster around
similar CTR values) means it contributes little linear signal to the product, so
impressions dominates clicks_leak's behavior. impressions and ctr_pct themselves
are essentially uncorrelated (r=-0.014), confirming they were non-redundant before
the leak was introduced. The leakage here is real but partial, not a clean 1:1 copy.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [13]:
excluded_summary = {
    "trend_direction, trend_pct, is_declining_label": "Label-derived; not present in this warehouse schema at all (starter-CSV only), but excluded on principle — would leak decision-relevant signal into clustering.",
    "is_published, is_deleted": "Product/status flags, not performance signals — clustering on these would group by an existing decision, not discovered structure.",
    "keyword_hash_id, url_hash_id, client_hash_id, content_hash_id": "Identifiers — used only for joining/grouping, never as model inputs (they carry no measurable pattern, just labels for rows).",
    "search_volume, competition, cpc": "Keyword-level SEO metadata, not observed page performance. Left out to keep this feature set focused on behavior FlyRank actually measured happening, not upstream keyword characteristics — a reasonable future addition with its own justification.",
    "content_updated_date": "Excluded after verification — 88.5% of rows had implausible future-dated values, meaning this column does not represent 'last edited' as its name implies. Replaced with content_created_date.",
    "clicks_leak": "Deliberately added and removed — mathematically derived from impressions and ctr_pct already in the feature set (Section 3 leakage experiment).",
}
for col, reason in excluded_summary.items():
    print(f"- {col}: {reason}")

- trend_direction, trend_pct, is_declining_label: Label-derived; not present in this warehouse schema at all (starter-CSV only), but excluded on principle — would leak decision-relevant signal into clustering.
- is_published, is_deleted: Product/status flags, not performance signals — clustering on these would group by an existing decision, not discovered structure.
- keyword_hash_id, url_hash_id, client_hash_id, content_hash_id: Identifiers — used only for joining/grouping, never as model inputs (they carry no measurable pattern, just labels for rows).
- search_volume, competition, cpc: Keyword-level SEO metadata, not observed page performance. Left out to keep this feature set focused on behavior FlyRank actually measured happening, not upstream keyword characteristics — a reasonable future addition with its own justification.
- content_updated_date: Excluded after verification — 88.5% of rows had implausible future-dated values, meaning this column does not represent 'last edited' a

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.